In [7]:
import os 
import numpy as np 
import pandas as pd 
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import kurtosis,skew
# Logistic ,KNN,SVM,RF
from sklearn.model_selection import train_test_split



In [3]:
# Example fix for data loading
os.makedirs("plots",exist_ok=True)
os.makedirs("model",exist_ok=True)


In [15]:
DATA_DIR=r"C:\Users\Asus\Downloads\PredictiveAI\bearing-dataset"
FEATURE_CSV="bearing_features.csv"
def extract_feature():
    if os.path.exists(FEATURE_CSV):
        print(f"[*] Found feature at '{FEATURE_CSV}'. Loading....")
        return pd.read_csv(FEATURE_CSV)
    print("[*] Extracting features  from raw bearing files")
    start_time=time.time()
    files=sorted([f for f in os.listdir(DATA_DIR) if f.startwidth(2004.02) ])
    if not files:
        raise FileNotFoundError(f" No test 2 files starting with (2004.02) found in {DATA_DIR}")
    print(f"[*] Found {len(files)} files to process")
    feature_list=[]
    for idx,file in enumerate(file):
        file_path=os.path.join(DATA_DIR,file)
        try:
            part=file.split('.')
            timestamp=datetime(
                year=int(part[0]),
                month=int(part[1]),
                day=int(part[2]),
                hour=int(part[3]),
                minute=int(part[4]),
                second=int(part[5])
            )
        except Exception:
            timestamp=np.nan


        try:
            df=pd.read_csv(file_path,sep="/t",header=None)
            if df.shape[1]<4:
                continue
            row_features={
                'filename':file,
                'timestamp':timestamp,
                'file_index':idx
            }
            for i in range(4):
                col_data=df[i].values
                mean_val=np.mean(col_data)
                std_val=np.std(col_data)
                rms_val=np.sqrt(np.mean(col_data**2))
                peak_val=np.max(np.abs(col_data))

                row_features[f'B{i+1}_mean']=mean_val
                row_features[f'B{i+1}_std']=std_val
                row_features[f'B{i+1}_rms']=rms_val
                row_features[f'B{i+1}_peak']=peak_val
                row_features[f'B{i+1}_kurtosis']=kurtosis(col_data)
                row_features[f'B{i+1}_skew']=skew(col_data)
                row_features[f'B{i+1}_crest_factor']=peak_val/rms_val if rms_val>0 else 0
                row_features[f'B{i+1}_shape_factor']=rms_val/np.mean(np.abs(col_data)) if np.mean(np.abs(col_data))>0 else 0
            feature_list.append(row_features)
        except Exception as e:
            print("f[!] Error processing file {file}:{e}")
        if(idx+1)%100==0:
            print(f"Processed {idx+1}/{len(files)}...files")
    df_features=df.DataFrame(feature_list)
    df_features.to_csv(FEATURE_CSV,index=False)
    print(f"[+] Feature extraction completed in {time.time()-start_time:.2f} seconds")
    return df_features
    


In [16]:
def run_eda(df):
    """Generates and saves exploratory plots showing bearing degradation."""
    print("[*] Generating EDA...")
    plt.figure(figsize=(12,6))
    for i in range(1,5):
        plt.plot(df['file_index'], df[f'B{i}_rms'], label=f'Bearing {i}')
    # Plot RMS trends for all bearings
    plt.axvline(x=700,color='r',linestyle='--',label='Fault Threshold  (Approx)')
    plt.title('RMS value  Trend(Test 2)-Acceleration Degradation Overtime ')
    plt.xlabel('File index(Timeline)')
    plt.ylabel('RMS value')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('plots/bearing_rms_trend.png')
    plt.close()

    #plot for kurtosis
    plt.figure(figsize=(12,6))
    for i in range(1,5):
        plt.plot(df['file_index'],df[f'B{i}_kurtosis'],label=f'Bearing {i}')
    plt.axvline(x=700,color='r',linestyle='--',label='Fault Threshold{i}')
    plt.title('Kurtosis trend (Test-2) -Structural Wear Indicator')
    plt.xlabel('File index (timeline)')
    plt.ylabel('kurtosis')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('plots/bearing_kurtosis_trend.png')
    plt.close()

    #correlation heapmap between bearing RMS
    rms_cols=[f'B{i}' for i in range(1,5)]
    plt.figure(figsize=(8,6))
    sns.heatmap(df[rms_cols].corr(),annot=True,cmap='coolwarm',fmt=".2f")
    plt.title('Correlation Matrix of Bearing RMS Value')
    plt.tight_layout()
    plt.savefig('plots/bearing_correlation.png')
    plt.close()

    print("[+] EDA plots saved inside 'plots/' folder.")
    


In [35]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score,recall_score,accuracy_score,precision_score
from sklearn.metrics import confusion_matrix
import joblib
def train_and_evaluate(df):
    """Trains and compares Logistic Regression, KNN, SVM, and Random Forest."""
    print("[*] Preparing dataset for training...")
    # In Test 2, Bearing 1 fails at the end. 
    # Let's label the last 284 files (index >= 700) as faulty/degraded (1), and first 700 as healthy
    df['state']=(df['file_index']>=700).astype(int)
    feature_cols=[]
    for i in range(1,5):
        feature_cols.extend([f'B{i}_mean', f'B{i}_std', f'B{i}_rms', f'B{i}_peak', 
                             f'B{i}_kurtosis', f'B{i}_skew', f'B{i}_crest_factor', f'B{i}_shape_factor'])
    X=df[feature_cols]
    y=df['state']
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    scaler=StandardScaler()
    X_train_scaler=scaler.fit_transform(X_train)
    X_test_scaler=scaler.transform(X_test)
    models={
        'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),
        'KNN':KNeighborsClassifier(n_neighbors=5),
        'SVM':SVC(kernel='rbf',probability=True,random_state=42),
        'Random Forest':RandomForestClassifier(n_estimators=100,random_state=42)
    }
    results={}
    best_model_name=None
    best_f1=0
    best_model=None

    print("[*] Training model")
    fig,axes=plt.subplots(2,2,figsize=(12,10))
    axes=axes.flatten()
    for idx ,(name,clf) in enumerate(models.items()):
        clf.fit(X_train_scaler,y_train)
        y_pred=clf.predict(X_test_scaler)
        acc=accuracy_score(y_test,y_pred)
        prec=precision_score(y_test,y_pred,zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)   

        results[name]={
            'Accuracy':acc,
            'Precision':prec,
            'Recall':rec,
            'F1-Score':f1
        }
        if f1>best_f1:
            best_f1=f1
            best_model_name=name
            best_model=clf
        cm=confusion_matrix(y_test,y_pred)
        sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[idx],cbar=False)
        axes[idx].set_title(f'{name}\n Accuracy :{acc:.4f} | F1:{f1:.4f}')
        axes[idx].set_xlabel('Predicted Label')
        axes[idx].set_ylabel('True label')
    plt.suptitle('Confusion matrix comparison for bearing fault  classification ',fontsize=16)
    plt.tight_layout()
    plt.savefig('plots/model_confusion_matrices.png')
    plt.close()

    print("\n"+"="*70)
    print(f'{'Classifier Model':<25}|{'Accuracy':<10}|{'Prediction':<10}|{'F1-Score':<10}|{'Recall':<10}')
    for name ,metrics in results.items():
        print(f'{name:<25}|{metrics['Accuracy']:<10.2%}|{metrics['Precision']:<10.2%}|{metrics['Recall']:<10.2%}|{metrics['F1-Score']:<.4f}')
    print("="*70)
    print(f"\n[+] Best Model based on F1-Score: {best_model_name} (F1: {best_f1:.4f})")

    # Save best model & feature metadata
    joblib.dump(best_model,'model/best_bearing_model.joblib')
    joblib.dump(feature_cols,'model/feature_columns.joblib')
    print("[+] Saved best model to 'models/best_bearing_model.joblib'")
    



In [36]:
def main():
    print("="*60)
    print("   IMS BEARING  FAULT DIAGNOSIS MODEL PIPELINE")
    print("= ")
    print("="*60)

    # Feature extraction 
    df_feat=extract_feature()
    # exploratory data analysis
    run_eda=df_feat
    # Model training & evaluation 
    train_and_evaluate(df_feat)
    print("\n[+] Pipeline executed successfully! Check 'plots/' for visualizations.")
    print("="*60)
    
if __name__ == "__main__":
    main()


   IMS BEARING  FAULT DIAGNOSIS MODEL PIPELINE
= 
[*] Found feature at 'bearing_features.csv'. Loading....
[*] Preparing dataset for training...
[*] Training model

Classifier Model         |Accuracy  |Prediction|F1-Score  |Recall    
Logistic Regression      |96.95%    |98.11%    |91.23%    |0.9455
KNN                      |93.91%    |100.00%   |78.95%    |0.8824
SVM                      |97.46%    |98.15%    |92.98%    |0.9550
Random Forest            |98.98%    |100.00%   |96.49%    |0.9821

[+] Best Model based on F1-Score: Random Forest (F1: 0.9821)
[+] Saved best model to 'models/best_bearing_model.joblib'

[+] Pipeline executed successfully! Check 'plots/' for visualizations.
